In [ ]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import xgboost
from pathlib import Path

In [ ]:
path=Path('/content')
my_list=[pd.read_csv(df) for df in path.glob('*.csv')]

In [ ]:
df=pd.concat(my_list,ignore_index=True)

In [ ]:
len(df)


58440

In [ ]:
df.head()

,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,precipitation_hours,relative_humidity_2m_mean,surface_pressure_mean,cloud_cover_mean,dew_point_2m_mean,wind_speed_10m_max,wind_gusts_10m_max,wind_direction_10m_dominant,shortwave_radiation_sum,et0_fao_evapotranspiration,city,Unnamed: 0
0,2010-01-01,15.1,17.6,12.9,0.0,0.0,0.0,71,1015.8,32,9.8,10.5,19.8,231,8.94,1.69,jadida,NaN
1,2010-01-02,14.3,17.8,11.3,0.0,0.0,0.0,80,1014.5,28,10.7,9.7,16.6,146,12.53,1.94,jadida,NaN
2,2010-01-03,15.9,19.4,13.0,0.0,0.0,0.0,83,1014.2,90,12.9,14.9,26.3,187,10.99,1.83,jadida,NaN
3,2010-01-04,16.9,18.9,15.8,5.2,5.2,15.0,80,1008.5,38,13.4,36.2,65.5,235,9.04,2.13,jadida,NaN
4,2010-01-05,15.7,16.2,14.7,5.4,5.4,19.0,79,1009.3,62,12.0,28.6,58.7,255,6.85,1.76,jadida,NaN


In [ ]:
df.isna().sum()

,0
date,0
temperature_2m_mean,0
temperature_2m_max,0
temperature_2m_min,0
precipitation_sum,0
rain_sum,0
precipitation_hours,0
relative_humidity_2m_mean,0
surface_pressure_mean,0
cloud_cover_mean,0


In [ ]:
df.drop('Unnamed: 0',axis=1,inplace=True)

In [ ]:
df['date']=pd.to_datetime(df['date'])
df['date'].dtype

dtype('<M8[ns]')

In [ ]:
df['month']=df['date'].dt.month

In [ ]:
df['month_cos']=np.cos(2*np.pi*df['month']/12)
df['month_sin']=np.sin(2*np.pi*df['month']/12)

In [ ]:
list(df.columns)

['date',
 'temperature_2m_mean',
 'temperature_2m_max',
 'temperature_2m_min',
 'precipitation_sum',
 'rain_sum',
 'precipitation_hours',
 'relative_humidity_2m_mean',
 'surface_pressure_mean',
 'cloud_cover_mean',
 'dew_point_2m_mean',
 'wind_speed_10m_max',
 'wind_gusts_10m_max',
 'wind_direction_10m_dominant',
 'shortwave_radiation_sum',
 'et0_fao_evapotranspiration',
 'city',
 'month',
 'month_cos',
 'month_sin']

In [ ]:
df['temp_range']=(df['temperature_2m_max']-df['temperature_2m_min'])
df['dew_point_depression']=(df['temperature_2m_mean']-df['dew_point_2m_mean'])
df['win_dir_sin']=np.sin(np.deg2rad(df['wind_direction_10m_dominant']))
df['win_dir_cos']=np.cos(np.deg2rad(df['wind_direction_10m_dominant']))
doy=df['date'].dt.day_of_year
df['doy_sin']=np.sin(2*np.pi*doy/365.25)
df['doy_cos']=np.cos(2*np.pi*doy/365.25)

In [ ]:
g=df.groupby('city')
for k in (1,3,7):
    df[f'rain_lag_{k}']=g['rain_sum'].shift(k)
df['pressure_change_1d']=g['surface_pressure_mean'].diff()
df['humidity_lag_1']=g['relative_humidity_2m_mean'].shift()

In [ ]:
df['rained']=(df['precipitation_sum']>0.1).astype(int)

In [ ]:
for w in (3,7,30):
    df[f'rain_roll_{w}_sum']=g['rain_sum'].transform(lambda s:s.shift(1).rolling(w).sum())

In [ ]:
df=df.sort_values(['city','date'])
test=df.groupby('city',group_keys=False).apply(lambda g:g.tail(int (len(g)*0.2)))
df_train=df.drop(test.index)
df_test=df.loc[test.index]


/tmp/ipykernel_541/3581834503.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test=df.groupby('city',group_keys=False).apply(lambda g:g.tail(int (len(g)*0.2)))


In [ ]:
cols_to_drop=['precipitation_sum', 'precipitation_hours', 'month','date','rain_sum','rained']
df=df.dropna()
df['city']=df['city'].astype('category')
df=df.sort_values(['city','date'])
g=df.groupby('city')
df['rained']=g['rained'].shift(-1)
df.dropna(subset='rained',inplace=True)

/tmp/ipykernel_541/521335191.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g=df.groupby('city')


In [ ]:
X_train=df_train.drop(cols_to_drop,axis=1)
X_test=df_test.drop(cols_to_drop,axis=1)
y_train=df_train['rained']
y_test=df_test['rained']
X_train['city']=X_train['city'].astype('category')
X_test['city']=X_test['city'].astype('category')

In [ ]:
#We see that our dataset is imbalanced we must take it into account when creating our model:
df['rained'].value_counts()

,count
rained,
0.0,44261
1.0,13869


In [ ]:
from xgboost import XGBClassifier
v0=XGBClassifier(n_estimators=600,learning_rate=0.03,max_depth=6,subsample=0.8,colsample_bytree=0.8,enable_categorical=True,
                 scale_pos_weight=44261/13879,tree_method='hist',random_state=42)

In [ ]:
v0.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=600,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_pred=v0.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix
cm_0=confusion_matrix(y_test,y_pred)
cm_0

array([[7942,  864],
       [ 567, 2307]])

In [ ]:
def extract_metrics(y_test,y_pred):
    cm=confusion_matrix(y_test,y_pred)
    tn=cm[0][0]
    fp=cm[0][1]
    fn=cm[1][0]
    tp=cm[1][1]
    total=tn+fp+fn+tp
    accuracy=(tp+tn)/total if total>0 else 0
    precision=tp/(tp+fp) if (tp+fp)>0 else 0
    recall=tp/(tp+fn) if (tp+fn)>0 else 0
    f1_score=2*(precision*recall)/(precision+recall) if (precision+recall)>0 else 0
    return pd.DataFrame({'accuracy':[np.round(accuracy,3)],
                        'precision':[np.round(precision,3)],
                        'recall':[np.round(recall,3)],
                        'f1_score':[np.round(f1_score,3)]})

In [ ]:
extract_metrics(y_test,y_pred)

,accuracy,precision,recall,f1_score
0,0.877,0.728,0.803,0.763


In [ ]:
param_grid_regularized = {
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [1, 3, 7, 15],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.6, 0.75, 0.9],
    'colsample_bytree': [0.6, 0.75, 0.9],
    'reg_alpha': [0, 0.1, 1.0, 5.0],       # L1 regularization
    'reg_lambda': [0.5, 1.0, 5.0, 10.0],   # L2 regularization
    'gamma': [0, 0.1, 0.5, 1.0],           # Minimum loss reduction for split
    'scale_pos_weight': [1.0, 2.5, 3.19, 4.0]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV,TimeSeriesSplit
base_model=XGBClassifier(n_estimators=600,tree_method='hist',enable_categorical=True,random_state=42,eval_metric='logloss',device='cuda')
tscv=TimeSeriesSplit(n_splits=5)
search=RandomizedSearchCV(estimator=base_model,
                          param_distributions=param_grid_regularized,
                          n_iter=20,
                          cv=tscv,
                          scoring='f1',
                          random_state=42,
                          verbose=1,
                          n_jobs=1)
search.fit(X_train,y_train)
print(f"Best params:{search.best_params_}")
print(f"Best cross validation f1_score:{search.best_score_}")
best_model=search.best_estimator_
y_pred_best=best_model.predict(X_test)
extract_metrics(y_test,y_pred_best)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best params:{'subsample': 0.9, 'scale_pos_weight': 1.0, 'reg_lambda': 1.0, 'reg_alpha': 0, 'min_child_weight': 1, 'max_depth': 9, 'learning_rate': 0.03, 'gamma': 0, 'colsample_bytree': 0.9}
Best cross validation f1_score:0.771856738484308


,accuracy,precision,recall,f1_score
0,0.885,0.841,0.658,0.738
